# Multi-Turn Conversation

In [3]:
# Code from previous notebook 
import boto3

#Setting up the client object instance
client = boto3.client("bedrock-runtime", region_name = "us-west-2")

# Setting up the model ID -> Important to know the difference between model id and inference id
# model id can be available in a region but not in other, hence leverage inference profile
# Here to start with we are using model id
model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"

user_message = {
    "role": "user",
    "content": [
        { "text": "What is 4 +4 ?" }
    ]
}

response = client.converse(
    modelId=model_id , 
    messages=[user_message] # it can ingest a list of message
)

response['output']['message']['content'][0]['text']

'4 + 4 = 8'

In [2]:
# now let's try to make another call to anthropic to see how it respond
abs

In [4]:
## I am send another request just after previous one and see the response
## model is not aware of previous context

user_message = {
    "role": "user",
    "content": [
        { "text": "What if add 2 more to it?" }
    ]
}

response = client.converse(
    modelId=model_id , 
    messages=[user_message] # it can ingest a list of message
)

response['output']['message']['content'][0]['text']

'I don\'t have any previous context about what "it" refers to or what the original number or quantity was. Could you please provide more details about what you\'d like to add 2 to? For example:\n\n- A specific number\n- A calculation we were working on\n- A quantity in a problem\n- Something else entirely\n\nOnce you give me that context, I\'ll be happy to help you add 2 to it!'

### Now we need to provide the context to model by bundling the request response from past conversation

In [5]:
#Setting up the client object instance
multiturn_client = boto3.client("bedrock-runtime", region_name = "us-west-2")

# Setting up the model ID -> Important to know the difference between model id and inference id
# model id can be available in a region but not in other, hence leverage inference profile
# Here to start with we are using model id
model_id = "us.anthropic.claude-sonnet-4-20250514-v1:0"

In [18]:
## Since I will append messages from conversations I will create a variable of type list to store the message
messages = []

# Both function combined are making context for the model

# function take 'text' as input and format it in required format and add it to message list - this is for user input
def add_user_message(messages, text):
    user_message = {
        "role" : "user",
        "content" : [
            { "text" : text }
        ]
    }
    messages.append(user_message)

# function take 'text' as input and format it in required format and add it to message list - this is for model output
def add_assistant_message(messages, text):
    user_message = {
        "role" : "assistant",
        "content" : [
            { "text" : text }
        ]
    }
    messages.append(user_message)


def chat(messages):
    response = client.converse(
        modelId = model_id,
        messages=messages
    )
    return response['output']['message']['content'][0]['text']

In [19]:
add_user_message(messages, "what is 3+5?")
messages

[{'role': 'user', 'content': [{'text': 'what is 3+5?'}]}]

In [20]:
assitant_response = chat(messages)
assitant_response

'3 + 5 = 8'

In [21]:
add_assistant_message(messages, assitant_response)
add_user_message(messages, "what if I add 2 more?")
assitant_response = chat(messages)
assitant_response

'If you add 2 more to 8, you get:\n\n8 + 2 = 10'

### If you see above output it mantain context